In [3]:
!pip install transformers datasets accelerate evaluate peft scikit-learn triton


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00


In [4]:
import torch
import numpy as np
import random
from transformers import set_seed

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


# Описание датасета
https://huggingface.co/datasets/wangrongsheng/ag_news

AG is a collection of more than 1 million news articles. News articles have been gathered from more than 2000 news sources by ComeToMyHead in more than 1 year of activity.

In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Загрузка датасета
dataset_name = "ag_news"
# Возьмем подвыборку (2000 train и 500 test)
full_dataset = load_dataset(dataset_name)
train_dataset = full_dataset["train"].shuffle(seed=SEED).select(range(2000))
eval_dataset = full_dataset["test"].shuffle(seed=SEED).select(range(500))

# Информация о лейблах
num_labels = 4
id2label = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
label2id = {v: k for k, v in id2label.items()}

# 2. Выбор модели и токенизатора
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Препроцессинг
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

print("Dataset loaded and tokenized.")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset loaded and tokenized.


In [6]:
import pandas as pd
import random

def show_random_examples(dataset, num_examples=5):
    # Выбираем случайные индексы
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        picks.append(pick)

    # Берем подвыборку
    df = pd.DataFrame(dataset[picks])

    # Добавляем текстовую расшифровку метки класса
    if "label" in df.columns:
        df["label_name"] = df["label"].apply(lambda x: id2label[x])

    # Настройка, чтобы длинный текст не обрезался
    pd.set_option('display.max_colwidth', None)
    return df

print(f"Примеры данных (случайные {5} шт.):")
display(show_random_examples(train_dataset))

Примеры данных (случайные 5 шт.):


,text,label,label_name
0,"Mourners honour Iraq aid worker A memorial service is held in Ireland for Margaret Hassan, as the EU warns of effects for relief workers in Iraq.",0,World
1,"Ex-U.S. Cyber Security Chief Sees Curb on Phishing SAN FRANCISCO (Reuters) - A former White House Web security chief predicted on Wednesday that technology companies and law enforcers could soon stamp out most Internet ""phishing"" scams that aim to trick people into giving away personal and financial information.",3,Sci/Tech
2,Panthers #39; defense leads to another win Carl Krauser scored all but two of his 17 points at the free throw line to lead No. 11 Pittsburgh to a 70-51 victory over Memphis on Tuesday night in the Jimmy V Classic.,1,Sports
3,"Turkey defends plans to outlaw adultery Turkish Prime Minister Tayyip Erdogan has defended plans to outlaw adultery that have outraged women #39;s groups and raised eyebrows in the European Union, which Turkey aspires to join.",0,World
4,Moss Among Nine Vikings Fined for Bears Altercation NEW YORK (Sports Network) - The NFL handed down fines to 15 players involved in a scuffle between the Minnesota Vikings and Chicago Bears on Sept. 26.,1,Sports


In [ ]:
import evaluate
from transformers import TrainingArguments, Trainer, __version__
import time

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = accuracy.compute(predictions=predictions, references=labels)
    f1_score = f1.compute(predictions=predictions, references=labels, average="weighted")
    return {"accuracy": acc["accuracy"], "f1": f1_score["f1"]}

# Базовые аргументы
base_training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_dir='./logs',
    report_to="none",
    seed=SEED
)

DEBUG: transformers version: 4.57.3


In [ ]:
# Инициализация модели
model_baseline = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=num_labels, id2label=id2label, label2id=label2id
).to(device)

trainer_baseline = Trainer(
    model=model_baseline,
    args=base_training_args,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)
print("Evaluating Baseline (As-is)...")
baseline_results = trainer_baseline.evaluate()
print(f"Baseline Results: {baseline_results}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Evaluating Baseline (As-is)...


Baseline Results: {'eval_loss': 1.3846538066864014, 'eval_model_preparation_time': 0.0012, 'eval_accuracy': 0.248, 'eval_f1': 0.14142181067594942, 'eval_runtime': 2.4481, 'eval_samples_per_second': 204.237, 'eval_steps_per_second': 13.071}


In [ ]:
print("\n--- Starting Full Fine-Tuning ---")

# Перезагружаем модель "чистой"
model_full = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=num_labels, id2label=id2label, label2id=label2id
).to(device)

trainer_full = Trainer(
    model=model_full,
    args=base_training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

start_time = time.time()
trainer_full.train()
full_ft_time = time.time() - start_time

full_ft_results = trainer_full.evaluate()
print(f"Full Fine-Tuning Results: {full_ft_results}")
print(f"Training time: {full_ft_time:.2f} seconds")


--- Starting Full Fine-Tuning ---


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.406472,0.876000,0.876118
2,No log,0.384117,0.878000,0.877527
3,No log,0.376540,0.892000,0.891482


Full Fine-Tuning Results: {'eval_loss': 0.3765398859977722, 'eval_accuracy': 0.892, 'eval_f1': 0.8914817261448841, 'eval_runtime': 1.9033, 'eval_samples_per_second': 262.699, 'eval_steps_per_second': 16.813, 'epoch': 3.0}
Training time: 70.03 seconds


In [ ]:
print("\n--- Starting Linear Probing ---")

model_linear = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=num_labels, id2label=id2label, label2id=label2id
).to(device)

# Заморозка весов: Проходим по всем параметрам и отключаем градиенты
for name, param in model_linear.base_model.named_parameters():
    param.requires_grad = False

# Проверим, что обучаем только классификатор
trainable_params = sum(p.numel() for p in model_linear.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model_linear.parameters())
print(f"Trainable parameters: {trainable_params} || All params: {all_params} || %: {100 * trainable_params / all_params:.2f}%")

# Для Linear Probing часто нужен learning rate повыше, так как мы учим слой с нуля
linear_args = TrainingArguments(
    output_dir="./results_linear",
    learning_rate=1e-3, # Увеличили LR
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=SEED
)

trainer_linear = Trainer(
    model=model_linear,
    args=linear_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

start_time = time.time()
trainer_linear.train()
linear_time = time.time() - start_time

linear_results = trainer_linear.evaluate()
print(f"Linear Probing Results: {linear_results}")
print(f"Training time: {linear_time:.2f} seconds")


--- Starting Linear Probing ---


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable parameters: 593668 || All params: 66956548 || %: 0.89%


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.480442,0.850000,0.850048
2,No log,0.398782,0.872000,0.872134
3,No log,0.442079,0.862000,0.862658
4,0.372400,0.375012,0.876000,0.875706
5,0.372400,0.380868,0.872000,0.872163


Linear Probing Results: {'eval_loss': 0.38086846470832825, 'eval_accuracy': 0.872, 'eval_f1': 0.8721625986147635, 'eval_runtime': 1.7666, 'eval_samples_per_second': 283.026, 'eval_steps_per_second': 18.114, 'epoch': 5.0}
Training time: 46.58 seconds


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

print("\n--- Starting LoRA Training ---")

# 1. Загружаем базу
model_lora_base = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=num_labels, id2label=id2label, label2id=label2id
).to(device)

# 2. Конфигурация LoRA
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"]
)

# 3. Оборачиваем модель
model_lora = get_peft_model(model_lora_base, peft_config)
model_lora.print_trainable_parameters()

# Используем стандартный LR
lora_args = TrainingArguments(
    output_dir="./results_lora",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=SEED
)

trainer_lora = Trainer(
    model=model_lora,
    args=lora_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

start_time = time.time()
trainer_lora.train()
lora_time = time.time() - start_time

lora_results = trainer_lora.evaluate()
print(f"LoRA Results: {lora_results}")
print(f"Training time: {lora_time:.2f} seconds")


--- Starting LoRA Training ---


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 741,124 || all params: 67,697,672 || trainable%: 1.0948


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.391931,0.862000,0.861573
2,No log,0.394746,0.866000,0.865374
3,No log,0.375878,0.870000,0.869659


LoRA Results: {'eval_loss': 0.37587806582450867, 'eval_accuracy': 0.87, 'eval_f1': 0.8696591147653112, 'eval_runtime': 1.9191, 'eval_samples_per_second': 260.542, 'eval_steps_per_second': 16.675, 'epoch': 3.0}
Training time: 51.31 seconds


In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    "Method": ["Baseline (As-Is)", "Full Fine-Tuning", "Linear Probing", "LoRA"],
    "Accuracy": [
        baseline_results['eval_accuracy'],
        full_ft_results['eval_accuracy'],
        linear_results['eval_accuracy'],
        lora_results['eval_accuracy']
    ],
    "F1 Weighted": [
        baseline_results['eval_f1'],
        full_ft_results['eval_f1'],
        linear_results['eval_f1'],
        lora_results['eval_f1']
    ],
    "Time (sec)": [
        0,
        full_ft_time,
        linear_time,
        lora_time
    ]
})

print(results_df)

             Method  Accuracy  F1 Weighted  Time (sec)
0  Baseline (As-Is)     0.248     0.141422    0.000000
1  Full Fine-Tuning     0.892     0.891482   70.027278
2    Linear Probing     0.872     0.872163   46.584236
3              LoRA     0.870     0.869659   51.310650


# Вывод:

### Метрики качества:
*   Full Fine-Tuning показал наилучшую точность (0.892).
*   Linear Probing (0.872) и LoRA (0.870) уступили полному дообучению порядка 2 п.п.
*   Разница между методами невелика, что свидетельствует о высоком качестве исходных представлений (эмбеддингов) модели DistilBERT для данной задачи.


### Вычислительная эффективность:
*   Linear Probing - самый быстрый метод (46 с), так как обновляются веса только выходного слоя.
*   Full Fine-Tuning - самый медленный метод (70 с), требующий полного прохода градиентов.
*   LoRA (51 с) занимает промежуточное положение по времени, но существенно экономит VRAM за счет малого количества обучаемых параметров.


### Итог:
Использование облегченных методов обучения (Linear Probing, LoRA) на данном датасете позволяет обеспечить ~97-98% от качества полного дообучения при сокращении времени тренировки на 30-35%.
